<a href="https://colab.research.google.com/github/Saharochek98/Statistics-of-Stochastic-Processes/blob/main/02_advanced_stochastic_models_and_calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import stats
from scipy.stats import norm, gaussian_kde
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit

np.random.seed(42)

In [ ]:
class StochasticProcess:
    def __init__(self, params, T, Nx, Nt, init_state = None):
        self.params = params
        self.T = T
        self.Nx = Nx
        self.Nt = Nt
        self.init_state = init_state

        #Wiener Process by default
        self.bxt = lambda x: 0
        self.sigmaxt = lambda x: 1

    def sampler_init_state(self):
        return np.zeros(self.Nx)

    def EulerSolution(self):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.sampler_init_state()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            dwt = np.random.normal(0, 1, size = self.Nx) * np.sqrt(dt)
            x_data[i] = x_data[i - 1] + self.bxt(x_data[i - 1]) * dt + self.sigmaxt(x_data[i - 1]) * dwt
        return t_data, x_data

# Examples

Exercise 1. Use the stochastic integral Ito, $N=1000$ to simulate and plot a path with $X_0 = 10$ of
*   the Ornstein-Uhlenbeck process, $\mathbb{\theta} = (-5.0, 0.0, 3.5)$;
*   the Cox-Ingersoll-Ross process, $\mathbb{\theta} = (2.00, 0.20, 0.15)$;
*   the Black-Scholes-Merton process, $\mathbb{\theta} = (1.0, 0.2)$.


In [ ]:
class OrnsteinUhlenbeckProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

        # params = [theta, mu, sigma]
        self.bxt = lambda x: self.params[0] * (self.params[1] - x)
        self.sigmaxt = lambda x: self.params[2]

    def sampler_init_state(self):
        return self.params[1] * np.ones(self.Nx)


class CoxIngersollRossProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)

        kappa, theta, sigma = self.params
        self.lower = 0.0
        self.bxt = lambda x: kappa * (theta - x)
        self.sigmaxt = lambda x: sigma * np.sqrt(np.maximum(x, 0))

    def sampler_init_state(self):
        return np.full(self.Nx, self.params[1])


class BlackScholesMertonProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

        # params = [theta1, theta2]
        self.bxt = lambda x: self.params[0] * x
        self.sigmaxt = lambda x: self.params[1] * x

    def sampler_init_state(self):
        return np.ones(self.Nx)

In [ ]:
T = 1
Nx = 1
Nt = 1000

init_state = 10 * np.ones(Nx)

theta = 5
mu = 0
sigma = 3.5
params = [theta, mu, sigma]

OUProcess = OrnsteinUhlenbeckProcess(params, T, Nx, Nt, init_state)
t_data, x_data = OUProcess.EulerSolution()
plt.plot(t_data, x_data, label = 'OU process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

In [ ]:
theta1 = 2
theta2 = 0.2
theta3 = 0.15
params = [theta1, theta2, theta3]

CIRProcess = CoxIngersollRossProcess(params, T, Nx, Nt, init_state)
t_data, x_data = CIRProcess.EulerSolution()
plt.plot(t_data, x_data, label = 'CIR process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

In [ ]:
theta1 = 1.0
theta2 = 0.2
params = [theta1, theta2]

BSMProcess = BlackScholesMertonProcess(params, T, Nx, Nt, init_state)
t_data, x_data = BSMProcess.EulerSolution()
plt.plot(t_data, x_data, label = 'BSM process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

Exercise 2. Compute a confidence interval of the CIR process from the previous task. Plot the paths and a confidence interval.

In [ ]:
def zeta(sigma, theta, t, s):
    return sigma**2 / (4 * theta) * (1 - np.exp(-theta * (t - s)))

def CIRbounds(alpha_interval, params, t, s, xs):
    theta1, theta2, theta3 = params[0], params[1], params[2]
    theta = theta2
    beta = theta1 / theta2
    sigma = theta3

    k = 4  / sigma**2 * theta * beta
    lambd = 4 * theta / sigma**2  * np.exp(-theta * (t - s)) / (1 - np.exp(-theta * (t - s))) * xs
    ksi = norm.ppf(1 - alpha_interval / 2)

    b1 = zeta(sigma, theta, t, s) * (k + lambd - ksi * np.sqrt(2 * (k + 2 * lambd)))
    b2 = zeta(sigma, theta, t, s) * (k + lambd + ksi * np.sqrt(2 * (k + 2 * lambd)))
    return [b1, b2]


In [ ]:
theta1 = 2
theta2 = 0.2
theta3 = 0.15
params = [theta1, theta2, theta3]

Nx = 20

x0 = 10
init_state = x0 * np.ones(Nx)
alpha_interval = 0.05

CIRProcess = CoxIngersollRossProcess(params, T, Nx, Nt, init_state)
t_data, x_data = CIRProcess.EulerSolution()
for k in range(0, Nx):
    plt.plot(t_data, x_data[:,k])

low_bound_data = np.zeros_like(t_data)
up_bound_data = np.zeros_like(t_data)

for i in range(1, len(t_data)):
    bounds = CIRbounds(alpha_interval, params, t_data[i], 0, x0)
    low_bound_data[i] = bounds[0]
    up_bound_data[i] = bounds[1]
low_bound_data[0] = x0
up_bound_data[0] = x0
plt.plot(t_data, up_bound_data, label = f"upper bound")
plt.plot(t_data, low_bound_data, label = f"lower bound")

plt.title(f"CIR process, 1 - alpha = {1-alpha_interval}, N = {Nx} trajectories")
plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

Exercise 3. For the Ornstein-Uhlenbeck process
$$
dX_t = \theta(\mu - X_t)dt + \sigma dW_t, \quad \theta = 1, \quad \mu = 1.2, \quad \sigma = 0.3
$$
generate three paths: $X_0 = 0.0, 2.0, \mathcal{N}\left(\mu, \frac{\sigma^2}{2\theta}\right)$ and plot in graph.


In [ ]:
T = 1
Nx = 1
Nt = 1000

theta = 1
mu = 1.2
sigma = 0.3
params = [theta, mu, sigma]

x0 = 0.0
init_state = x0 * np.ones(Nx)
OUProcess = OrnsteinUhlenbeckProcess(params, T, Nx, Nt, init_state)

t_data, x_data = OUProcess.EulerSolution()
plt.plot(t_data, x_data, label = f'OU process, x0 = {x0}')

x0 = 2.0
init_state = x0 * np.ones(Nx)
OUProcess = OrnsteinUhlenbeckProcess(params, T, Nx, Nt, init_state)

t_data, x_data = OUProcess.EulerSolution()
plt.plot(t_data, x_data, label = f'OU process, x0 = {x0}')


init_state = np.random.normal(mu, sigma**2/(2 * theta), size = Nx)
OUProcess = OrnsteinUhlenbeckProcess(params, T, Nx, Nt, init_state)

t_data, x_data = OUProcess.EulerSolution()
plt.plot(t_data, x_data, label = f'OU process, x0 ~ Normal')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

# Exercises

Exercise 1.  Design a process that stays in the interval $[0, 1]$ and mean-reverts around $1/2$, generate paths for different $\theta$ and plot their in graph. Prove your solution.


Hint
$$
\mu(x) = \theta (1/2 - x), \quad \sigma(x) = \sigma \cdot x (1-x).
$$

That is
$$
dx = \theta (1/2 - x )dt + \sigma \cdot x (1-x) dW.
$$
Note: diffusion goes to zero at boundaries $\sigma(0) = \sigma(1) = 0$ and
mean-reverts  $⇒$ always stay in $[0, 1]$.

In [ ]:
def DesingProcess(theta=0, n=10):
    # Parameters
    T = 1.0
    Nt = 1000
    sigma = 0.05

    dt = T / Nt
    t = np.linspace(0, T, Nt)

    x0 = 0.5
    x_data = np.zeros(Nt)
    x_data[0] = x0


    plt.figure(figsize=(15,5))
    for theta in np.arange(10, 100, 40):
        for i in range(1, Nt):
            x_prev = x_data[i - 1]
            drift = theta * (0.5 - x_prev) * dt
            diffusion = sigma * x_prev * (1 - x_prev) * np.sqrt(dt) * np.random.normal()
            x_data[i] = x_prev + drift + diffusion

        plt.plot(t, x_data, label=f'theta: {theta}')
        plt.axhline(y=0.5, color='r', linestyle='--')
        plt.xlabel('Time')
        plt.ylabel('x(t)')
        plt.title('Process that stays in the interval  [0,1]  and mean-reverts around  1/2')
        plt.legend()
        plt.grid(True)
    plt.show()

DesingProcess()

Exercise 2. Using the class template from Task 1, create classes and plot graphs for the following processes:

1.   Modified CIR model;
2.   Chan-Karolyi-Longstaff-Sanders family of models;
1.   Nonlinear mean reversion Ait-Sahalia model;
2.   Jacobi diffusion process;
1.   Ahn and Gao (inverse of Feller’s square root) model;
2.   Radial Ornstein-Uhlenbeck process.

Provide examples for illustration the results on graphs.





In [ ]:
class ModifiedCIRProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        # params = [theta, mu, sigma]
        self.bxt = lambda x: self.params[0] * (self.params[1] - x)
        self.sigmaxt = lambda x: self.params[2] * np.sqrt(np.maximum(x, 0))

    def sampler_init_state(self):
        return np.full(self.Nx, self.params[1])

In [ ]:
class CKLSProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        # params = [theta, mu, sigma, alpha]
        self.bxt = lambda x: self.params[0] * (self.params[1] - x)
        self.sigmaxt = lambda x: self.params[2] * x ** self.params[3]

    def sampler_init_state(self):
        return np.full(self.Nx, self.params[1])

In [ ]:
class AitSahaliaProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)

        alpha_m1, alpha_0, alpha_1, alpha_2, sigma, rho = self.params

        self.lower = 1e-8
        self.bxt = lambda x: (
            alpha_m1 / np.maximum(x, 1e-8)
            + alpha_0
            + alpha_1 * x
            + alpha_2 * x**2
        )

        self.sigmaxt = lambda x: (
            sigma * np.maximum(x, 0)**rho
        )

    def sampler_init_state(self):
        return np.full(self.Nx, 1.5)

In [ ]:
class JacobiProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)

        kappa, theta, sigma = self.params

        self.lower = 0.0
        self.upper = 3.0

        self.bxt = lambda x: kappa * (theta - x)

        self.sigmaxt = lambda x: sigma * np.sqrt(
            np.maximum(
                (x - self.lower) * (self.upper - x),
                0.0
            )
        )

    def sampler_init_state(self):
        return np.full(self.Nx, self.params[1])

In [ ]:
class AhnGaoProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        # params = [theta, mu, sigma]
        self.bxt = lambda x: self.params[0] * (self.params[1] - x)
        self.sigmaxt = lambda x: self.params[2] * np.sqrt(np.maximum(x, 0))

    def sampler_init_state(self):
        return np.full(self.Nx, self.params[1])

In [ ]:
class RadialOUProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        # params = [theta, sigma]
        self.bxt = lambda x: -self.params[0] * x
        self.sigmaxt = lambda x: self.params[1]

    def sampler_init_state(self):
        return np.zeros(self.Nx)  # Starting at the origin

In [ ]:
# Common parameters for the models
T = 1.0
Nx = 1
Nt = 1000

# Parameters for each model
params_cir = [0.1, 0.05, 0.1]  # Modified CIR
params_ckls = [0.1, 0.05, 0.1, 0.5]  # CKLS
params_ait = [0.5, 0.1, -2.0, 0.0, 0.3, 1.5]  # Ait-Sahalia
params_jacobi = [0.1, 0.05, 0.1]  # Jacobi
params_ahn = [0.1, 0.05, 0.1]  # Ahn and Gao
params_radial_ou = [15, 0.5]  # Radial OU

# Create instances of each process
processes = [
    ModifiedCIRProcess(params_cir, T, Nx, Nt),
    CKLSProcess(params_ckls, T, Nx, Nt),
    AitSahaliaProcess(params_ait, T, Nx, Nt),
    JacobiProcess(params_jacobi, T, Nx, Nt),
    AhnGaoProcess(params_ahn, T, Nx, Nt),
    RadialOUProcess(params_radial_ou, T, Nx, Nt)
]

# Plotting each process
plt.figure(figsize=(15, 7))

for i, process in enumerate(processes):
    t_data, x_data = process.EulerSolution()
    plt.subplot(3, 2, i + 1)
    plt.plot(t_data, x_data, label=process.__class__.__name__)
    plt.title(process.__class__.__name__)
    plt.xlabel('Time')
    plt.ylabel('x(t)')
    plt.grid()
    plt.legend()

plt.tight_layout()
plt.show()

**Example: Regime-Switching CIR Process Simulation**

Simulate a composite trajectory using the Cox-Ingersoll-Ross (CIR) process defined by:
$$
dX_t = (6 - 2X_t)dt + \theta \sqrt{X_t}dW_t, \quad X(0)=1,
$$
where the volatility parameter switches regimes: $\theta = \theta_1 = 1$ for $t < \tau_0 = 0.6$, and $\theta = \theta_2$ for $t \geq \tau_0$, with $t \in (0, 1)$.

**Tasks:**
1. Plot the simulated trajectory and the change-point detection statistic $|\hat{D}_k|$.
2. Mark the estimated change point $k_0$ on the trajectory plot.
3. Compare the estimated change point with the true value $\tau_0$.

In [ ]:
class CIRProcessSimulation:
    def __init__(self, n=1000, T=1.0, tau0=0.6, theta1=1.0, theta2=2.0, X0=1.0):
        self.n = n
        self.T = T
        self.tau0 = tau0
        self.theta1 = theta1
        self.theta2 = theta2
        self.X0 = X0
        self.Δt = self.T / self.n
        self.t = np.linspace(0, self.T, self.n + 1)
        self.X = np.zeros(self.n + 1)
        self.X[0] = self.X0
        self.D_k = None

    def simulate(self):
        ΔW = np.sqrt(self.Δt) * np.random.randn(self.n)
        for i in range(self.n):
            theta = self.theta1 if self.t[i] < self.tau0 else self.theta2
            sqrt_Xi = np.sqrt(max(self.X[i], 0))
            self.X[i+1] = self.X[i] + (6 - 2 * self.X[i]) * self.Δt + theta * sqrt_Xi * ΔW[i]
            self.X[i+1] = max(self.X[i+1], 0)

    def estimate_change_point(self):
        Z = ((self.X[1:] - self.X[:-1]) - (6 - 2 * self.X[:-1]) * self.Δt) / np.sqrt(self.Δt)
        Z_squared = Z**2
        S_k = np.cumsum(Z_squared)
        S_n = S_k[-1]
        k_values = np.arange(1, self.n + 1)
        self.D_k = (S_k / S_n) - (k_values / self.n)
        k0 = np.argmax(np.abs(self.D_k))
        t0_estimate = self.t[k0 + 1]
        return t0_estimate

    def plot_results(self, t0_estimate):
        fig, axs = plt.subplots(2, 1, figsize=(16, 8))

        # Plot process
        axs[0].plot(self.t, self.X, color='green', label='CIR process')
        axs[0].axvline(x=self.tau0, color='red', linestyle='--', label=f'True change point= {tau0:.4f}')
        axs[0].set_xlabel('t', fontsize=12)
        axs[0].set_ylabel('X(t)', fontsize=12)
        axs[0].set_title('CIR process', fontsize=14)
        axs[0].legend()
        axs[0].grid(True)

        # Plot estimation
        axs[1].plot(self.t[1:], np.abs(self.D_k), color='green', label='Staticstic |D_k|')
        axs[1].axvline(x=t0_estimate, color='blue', linestyle='--', label=f'Estimated change point= {t0_estimate:.4f}')
        axs[1].axvline(x=self.tau0, color='red', linestyle='--', label=f'True change point= {tau0:.4f}')
        axs[1].set_xlabel('t', fontsize=12)
        axs[1].set_ylabel('Staticstic |D_k|', fontsize=12)
        axs[1].set_title('|D_k|', fontsize=14)
        axs[1].legend()
        axs[1].grid(True)

        plt.tight_layout()
        plt.show()

tau0 = 0.6
simulation = CIRProcessSimulation()
simulation.simulate()
t0_estimate = simulation.estimate_change_point()

simulation.plot_results(t0_estimate)

print(f"True: t₀ = {simulation.tau0}")
print(f"Estimated: y₀ = {t0_estimate:.4f}")

# Tasks

**Task 1: Verification of CIR Process Moments (Sections 2.3, 8.1)**

Analytical formulas exist for calculating moments of any order for the CIR model.
1. Based on the formula (Jafari, 2017), compute the theoretical first and second moments $\mathbb{E}\{X(t)\}$ and $\mathbb{E}\{X^2(t)\}$ for given parameters $\kappa, \theta, \sigma$.
2. Generate $M=1000$ trajectories and compute the sample mean and variance at time $T$. Compare them with the theoretical values.
3. Draw conclusions.

In [ ]:
def theoretical_moments_cir(params, t, x0):
    kappa, theta, sigma = params
    mean = theta + (x0 - theta) * np.exp(-kappa * t)
    var = (x0 * sigma**2 / kappa) * (np.exp(-kappa * t) - np.exp(-2 * kappa * t)) + \
          (theta * sigma**2 / (2 * kappa)) * (1 - np.exp(-kappa * t))**2

    mean2 = mean**2 + var
    return mean, mean2, var

kappa = 2.0
theta = 0.5
sigma = 0.3
params = [kappa, theta, sigma]

x0 = 0.3
T = 1.0
Nt = 1000
M = 1000


process = CoxIngersollRossProcess(params, T, M, Nt, x0)
t, X = process.EulerSolution()
mean_t, mean2_t, var_t = theoretical_moments_cir(params, T, x0)

X_T = X[-1, :]
mean_e = np.mean(X_T)
mean2_e = np.mean(X_T**2)
var_e = np.var(X_T, ddof=1)

print("Moment | Theoretical | Empirical | Deviation %")
print(f"E[X(T)]  | {mean_t:.4f}      | {mean_e:.4f}     | {abs(mean_e-mean_t)/mean_t*100:.2f}%")
print(f"E[X^2(T)]| {mean2_t:.4f}      | {mean2_e:.4f}     | {abs(mean2_e-mean2_t)/mean2_t*100:.2f}%")
print(f"Var(X(T))| {var_t:.4f}      | {var_e:.4f}     | {abs(var_e-var_t)/var_t*100:.2f}%")


### Conclusion:
There is a strong agreement between the theoretically derived and empirically calculated moments, validating the correctness of the simulation.

**Task 2: Properties of the Black-Scholes-Merton Model (Sections 2.2, 8.4)**

The conditional density of the BSM process is log-normal.
1. Set parameters $\theta_1=16\%$ (risk-free rate) and $\theta_2=30\%$ (volatility), and simulate the geometric Brownian motion.
2. Plot a histogram of the log-returns $\log(X_T/X_0)$ and verify its fit to a normal distribution with parameters $\mu=(\theta_1 - \theta_2^2/2)T$ and $\sigma^2 = \theta_2^2 T$.
3. Draw conclusions.

In [ ]:
theta1 = 0.16
theta2 = 0.3
T = 1.0
Nt = 1000
M = 10000
X0 = 1.0

dt = T / Nt
t = np.linspace(0, T, Nt+1)
X = np.zeros((Nt+1, M))
X[0, :] = X0

for i in range(1, Nt+1):
    dW = np.random.normal(0, np.sqrt(dt), M)
    X[i, :] = X[i-1, :] * (1 + theta1*dt + theta2*dW)

X_T = X[-1, :]
log_returns = np.log(X_T / X0)
mu_t = (theta1 - theta2**2/2) * T
sigma_t = theta2 * np.sqrt(T)

emp_mean = np.mean(log_returns)
emp_var = np.var(log_returns, ddof=1)
emp_std = np.std(log_returns, ddof=1)

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(log_returns, bins=50, density=True, alpha=0.7, color='b')
x_norm = np.linspace(log_returns.min(), log_returns.max(), 100)
ax.plot(x_norm, stats.norm.pdf(x_norm, mu_t, sigma_t), 'r-', linewidth=2, label='Normal PDF')
ax.axvline(mu_t, color='r', linestyle='--', linewidth=2, label='Theoretical Mean')
ax.axvline(emp_mean, color='g', linestyle='-', linewidth=2, label='Empirical Mean')
ax.set_xlabel('log(X_T/X0)')
ax.set_ylabel('Density')
ax.set_title(f'Distribution of Log-Returns ({M} trajectories)')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Theoretical: μ = {mu_t:.4f}, σ = {sigma_t:.4f}")
print(f"Empirical:   μ = {emp_mean:.4f}, σ = {emp_std:.4f}")
print(f"Mean deviation: {abs(emp_mean - mu_t)/abs(mu_t)*100:.2f}%")
print(f"Variance deviation: {abs(emp_var - sigma_t**2)/sigma_t**2*100:.2f}%")

### Conclusion:
The numerical solution aligns perfectly with the analytical one. The log-returns are indeed normally distributed with the parameters predicted by the theory.

**Task 3: Stochastic Volatility Model (BSM + CIR)**

1. Use the CIR model to generate a volatility trajectory $\sigma(t)$ with parameters ensuring process positivity (Feller condition).
2. Pass $\sqrt{\sigma(t)}$ from step 1 into the `BlackScholesMertonProcess`.
3. Visualize using two subplots: top graph for volatility (CIR), bottom graph for asset price (modified BSM).
4. Save the simulation results to a `.csv` file.
5. Draw conclusions.


In [ ]:
T = 1
Nt = 1000
Nx = 1
X0 = 1

theta1 = 0.09
theta2 = 2
theta3 = 1.2
init_vol_sq = 0.1

thetaB=0.16

cir_params = [theta1, theta2, theta3]
cir_process = CoxIngersollRossProcess(cir_params, T, Nx, Nt, init_vol_sq)
t, vol_sq = cir_process.EulerSolution()
vol_sq = vol_sq.flatten()
volatility = np.sqrt(np.maximum(vol_sq, 0))

dt = T / Nt
X = np.zeros(Nt + 1)
X[0] = X0

for i in range(1, Nt + 1):
    dW = np.random.normal(0, np.sqrt(dt))
    X[i] = X[i-1] * (1 + thetaB * dt + volatility[i-1] * dW)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(t, volatility, 'b-', linewidth=1)
ax1.set_xlabel('t')
ax1.set_ylabel('Volatility')
ax1.set_title('Volatility Trajectory (CIR)')

ax2.plot(t, X, 'r-', linewidth=1)
ax2.set_xlabel('t')
ax2.set_ylabel('Asset Price')
ax2.set_title('Asset Price (Modified BSM)')
plt.tight_layout()
plt.show()

results = pd.DataFrame({
    'time': t,
    'volatility': volatility,
    'asset_price': X
})
results.to_csv('vol.csv', index=False)

### Conclusions:
1. The CIR process successfully generates strictly positive volatility.
2. The asset price dynamically reacts to volatility fluctuations.
3. CIR parameters were chosen to satisfy the Feller condition ($2\kappa\theta > \sigma_{cir}^2$), ensuring positivity.

**Task 4: Regime Switching Trajectory**

1. Simulate an Ornstein-Uhlenbeck (OU) process on $t \in [0, 0.5]$ with parameters ensuring mean reversion to $\mu=1.2$.
2. At $t=0.5$ (change point), switch the system dynamics to the Black-Scholes-Merton model, using the final value of the OU process as the initial state for $t \in [0.5, 1.0]$.
3. Concatenate the two trajectories into a single path and plot it, marking the regime change with a vertical line.
4. Save the results to a `.csv` file.
5. Draw conclusions. Explain how the trajectory properties change.

In [ ]:
Nx = 1
Nt = 1000
T1 = 0.5
T2 = 0.5

ou_params = [5.0, 1.2, 0.5]

bsm_params = [0.1, 0.4]
ou_process = OrnsteinUhlenbeckProcess(ou_params, T1, Nx, Nt)
t1, x1 = ou_process.EulerSolution()
bsm_process = BlackScholesMertonProcess(bsm_params, T2, Nx, Nt, x1[-1])
t2, x2 = bsm_process.EulerSolution()


t_full = np.concatenate([t1, t1[-1] + t2[1:]])
x_full = np.concatenate([x1, x2[1:]])

df = pd.DataFrame({
    'time': t_full,
    'value': x_full.flatten()
})
df.to_csv('switch.csv', index=False)

plt.figure(figsize=(12, 6))
plt.plot(t_full, x_full, 'b-', linewidth=1.5)
plt.axvline(x=0.5, color='r', linestyle='--', linewidth=2, label='Regime Change')
plt.xlabel('t')
plt.title('Regime Switching Trajectory')
plt.legend()
plt.show()

### Conclusion:
1. **OU Phase ($t \in [0, 0.5]$):** The trajectory exhibits mean-reverting behavior, oscillating around $\mu=1.2$.
2. **BSM Phase ($t \in [0.5, 1.0]$):** After the regime change, the process becomes a Geometric Brownian Motion. Changes are now proportional to the current value, leading to exponential growth or decay depending on the drift sign.
*Summary:* Transition from a stationary mean-reverting process to a non-stationary process where variance grows over time.

**Task 5: Comparison and Calibration of Mean-Reverting Models**

1. Consider the following models: Ornstein–Uhlenbeck (OU), Cox–Ingersoll–Ross (CIR), Modified CIR, Ait–Sahalia, and Jacobi diffusion. For each, specify: state space, parameter conditions for strict positivity, and stationary distribution (if it exists).
2. Generate 500 trajectories for each model using: $\kappa = 2.0, \theta = 1.0, X_0 = 1.5, T = 2.0, \Delta t = 0.001, \sigma = 0.3$. (For Ait-Sahalia: $\alpha_{-1}=0.5, \alpha_0=0.1, \alpha_1=-2, \alpha_2=0, \rho=1.5$).
3. For each model, estimate: sample mean and variance at $T$, sample mean-reversion speed, and the probability of exceeding the threshold $X_t > 2.0$ on $[0, T]$.
4. Plot 10 trajectories per model and a separate graph of sample means over time for all models.
5. Draw conclusions.

In [ ]:
Nx = 500
Nt = 2000
T = 2.0
dt = T / Nt

kappa = 2.0
theta = 1.0
sigma = 0.3
X0 = 1.5
threshold = 2.0

as_params = [0.5, 0.1, -2.0, 0.0, sigma, 1.5]

np.random.seed(42)

processes = {
    "OU": OrnsteinUhlenbeckProcess(
        [kappa, theta, sigma], T, Nx, Nt,
        np.full(Nx, X0)
    ),
    "CIR": CoxIngersollRossProcess(
        [kappa, theta, sigma], T, Nx, Nt,
        np.full(Nx, X0)
    ),
    "Modified CIR": ModifiedCIRProcess(
        [kappa, theta, sigma], T, Nx, Nt,
        np.full(Nx, X0)
    ),
    "Ait-Sahalia": AitSahaliaProcess(
        as_params, T, Nx, Nt,
        np.full(Nx, X0)
    ),
    "Jacobi": JacobiProcess(
        [kappa, theta, sigma], T, Nx, Nt,
        np.full(Nx, X0)
    )
}


def simulate(process, name):
    h = process.T / process.Nt
    t = np.linspace(0, process.T, process.Nt + 1)

    x = np.empty((process.Nt + 1, process.Nx))

    if process.init_state is None:
        x[0] = process.sampler_init_state()
    else:
        x[0] = process.init_state

    for i in range(process.Nt):
        prev = x[i]

        dW = np.sqrt(h) * np.random.randn(process.Nx)

        drift = process.bxt(prev)
        diffusion = process.sigmaxt(prev)

        next_x = prev + drift * h + diffusion * dW

        if name == "CIR":
            next_x = np.maximum(next_x, 0.0)

        elif name == "Modified CIR":
            next_x = np.maximum(next_x, 1e-8)

        elif name == "Ait-Sahalia":
            next_x = np.maximum(next_x, 1e-6)

        elif name == "Jacobi":
            next_x = np.clip(
                next_x, process.lower, process.upper
            )

        if not np.all(np.isfinite(next_x)):
            raise FloatingPointError(
                f"{name}: non-finite values at step {i + 1}"
            )

        x[i + 1] = next_x

    return t, x


def estimate_reversion(t, x):
    mean_t = np.mean(x, axis=1)
    x_initial = mean_t[0]

    def model(t, c, lam):
        return c + (x_initial - c) * np.exp(-lam * t)

    try:
        params, _ = curve_fit(
            model,
            t,
            mean_t,
            p0=[mean_t[-1], 2.0],
            bounds=([-np.inf, 0.0], [np.inf, np.inf]),
            maxfev=20000
        )
        return params[1], params[0]
    except (RuntimeError, ValueError):
        return np.nan, np.nan


results = {}
trajectories = {}

for name, process in processes.items():

    t, x = simulate(process, name)
    trajectories[name] = (t, x)

    x_T = x[-1]

    mean_T = np.mean(x_T)
    var_T = np.var(x_T, ddof=1)

    lambda_est, equilibrium_est = estimate_reversion(t, x)
    prob_exceed = np.mean(
        np.any(x > threshold, axis=0)
    )

    results[name] = {
        "Mean at T": mean_T,
        "Variance at T": var_T,
        "Reversion speed": lambda_est,
        "Estimated equilibrium": equilibrium_est,
        "P(max X > 2)": prob_exceed
    }


results_df = pd.DataFrame(results).T
print(results_df.round(5).to_string())
results_df.to_csv("model_comparison_results.csv")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, (t, x)) in enumerate(trajectories.items()):
    ax = axes[idx]

    for j in range(10):
        ax.plot(t, x[:, j], linewidth=0.8, alpha=0.8)

    ax.axhline(
        threshold,
        color="red",
        linestyle="--",
        label="Threshold = 2.0"
    )

    if name != "Ait-Sahalia":
        ax.axhline(
            theta,
            color="black",
            linestyle=":",
            label="theta = 1.0"
        )

    ax.set_title(name)
    ax.set_xlabel("Time")
    ax.set_ylabel("X(t)")
    ax.set_xlim(0, T)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].axis("off")

fig.suptitle(
    "10 Simulated Trajectories for Each Model",
    fontsize=16
)

plt.tight_layout()
plt.savefig("model_trajectories.png", dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(12, 7))

for name, (t, x) in trajectories.items():
    mean_t = np.mean(x, axis=1)
    plt.plot(t, mean_t, label=name, linewidth=2)

plt.axhline(
    theta,
    color="black",
    linestyle="--",
    label="theta = 1.0"
)

plt.xlabel("Time")
plt.ylabel("Sample Mean")
plt.title("Comparison of Sample Means Over Time")
plt.xlim(0, T)
plt.grid(alpha=0.3)
plt.legend()



### Conclusion

1. **State Space & Stationarity:** OU is unbounded and has a normal stationary distribution. CIR, Modified CIR, and Aït-Sahalia remain positive under appropriate parameter conditions, while Jacobi is bounded within [0, 3].

2. **Mean-Reversion Speed:** OU (1.955), CIR (2.064), Modified CIR (1.901), and Jacobi (2.036) exhibit estimated reversion speeds close to the theoretical value κ = 2.0.

3. **Threshold Exceedance:** No threshold exceedances were observed for OU, CIR, Modified CIR, or Aït-Sahalia. Jacobi showed a small exceedance probability of 0.2%. This reflects the results of 500 simulated trajectories.

4. **Aït-Sahalia:** Due to its nonlinear drift, Aït-Sahalia converges toward a lower equilibrium level (approximately 0.53), with the smallest variance (0.00167). Its estimated effective reversion speed is 2.878.

5. **Variance Comparison:** Jacobi exhibits the highest variance (0.04739), while OU, CIR, and Modified CIR have similar variances (approximately 0.02–0.025).

**Overall**, all five models demonstrate mean-reverting behavior, but differ in their volatility structures, equilibrium levels, and state-space constraints. The simulation results are consistent with their theoretical properties.

**Task 6: Two-Dimensional Stochastic Process**

1. Simulate the process:
$$ dX_t = \mu(X^1_t, X^2_t)dt + \sigma(X^1_t, X^2_t)dW_t, \quad \Theta=\{x \in \mathbf{R}^2 \mid x^2_1 + x^2_2=1\} $$
with piecewise drift $\mu$ and state-dependent diffusion $\sigma$ as defined in the code.
2. Plot the trajectory.
3. Draw conclusions.

In [ ]:
class StochasticProcessV2:
    def __init__(self, T, Nt, init_state):
        self.T = T
        self.Nt = Nt
        self.init_state = np.asarray(init_state, dtype=float)

        self.Nx = 2

    def mu(self, x):
        x1, x2 = x[0], x[1]
        r2 = x1**2 + x2**2

        if r2 <= 1:
            return np.array([-x1, x2])
        else:
            return np.array([1.0, 1.0])

    def sigma(self, x):
        x1, x2 = x[0], x[1]

        return (
            1 / (1 + x1**2 + x2**2)
        ) * np.array([
            [x1, 0.0],
            [x2, 0.0]
        ])

    def EulerSolution(self):
        dt = self.T / self.Nt

        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_prev = x_data[i - 1]

            dW = np.random.normal(
                0, np.sqrt(dt), size=2
            )

            x_new = (
                x_prev
                + self.mu(x_prev) * dt
                + self.sigma(x_prev) @ dW
            )

            x_data[i] = x_new

        return t_data, x_data

T = 10.0
Nt = 10000
init_state = np.array([1.10, 0.00])

process = StochasticProcessV2(T, Nt, init_state)
t, traj = process.EulerSolution()


fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(
    traj[:, 0],
    traj[:, 1],
    color="blue",
    linewidth=1,
    alpha=0.7,
    label="Trajectory"
)

circle = plt.Circle(
    (0, 0),
    1,
    fill=False,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Unit circle"
)

ax.add_patch(circle)

ax.scatter(
    init_state[0],
    init_state[1],
    color="green",
    s=100,
    marker="o",
    label="Start",
    zorder=5
)

ax.scatter(
    traj[-1, 0],
    traj[-1, 1],
    color="red",
    s=100,
    marker="o",
    label="End",
    zorder=5
)

ax.set_xlabel("X1(t)")
ax.set_ylabel("X2(t)")
ax.set_title("2D Stochastic Process Trajectory")

ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

r = np.sqrt(traj[:, 0]**2 + traj[:, 1]**2)

print("Initial state:", traj[0])
print("Final state:", traj[-1])
print("Minimum radius:", np.min(r))
print("Maximum radius:", np.max(r))
print("Fraction inside unit circle:", np.mean(r <= 1))
print("Fraction outside unit circle:", np.mean(r > 1))


### Conclusion

1. **Trajectory Behavior:** The process starts at (1.1, 0) and moves predominantly away from the origin, reaching approximately (11.28, 10.06) at T = 10.

2. **Drift Influence:** Outside the unit circle, the drift is constant at (1, 1), causing both coordinates to increase over time.

3. **Boundary Behavior:** The trajectory remains outside the unit circle for approximately 99.98% of the simulated time points, with only brief excursions inside.

4. **Diffusion Effect:** The state-dependent diffusion introduces random fluctuations, but its influence decreases as the distance from the origin increases.

**Overall**, the simulation demonstrates that the outward drift dominates the long-term behavior, while the unit circle acts as a boundary between two drift regimes rather than a reflecting or absorbing barrier.